# bloch_dispersion.ipynb — Classical Bloch Dispersion for Meta-Impactor Lattice

**Approach 1 of 2 — Periodic (Bloch) Lattice assumption**

## Physical assumption

The lattice is **infinite and perfectly periodic**.  Applying the Bloch condition
$x_{n\pm1}(t) = x_n(t)\,e^{\pm ik}$ reduces the chain EOM
$$M\ddot{x}_n + c\dot{x}_n + K(2x_n - x_{n-1} - x_{n+1}) = 0$$
to a **single-cell oscillator**:
$$M\ddot{x} + c\dot{x} + K_b(k)\,x = 0, \qquad K_b(k) = 2K(1-\cos k)$$
with the **same free-flying impactor** dynamics and impact condition $|x-y|=D$.

The linear (no-impactor) dispersion is the exact solution:
$$\omega_\mathrm{lin}(k) = 2\sqrt{K/M}\,|\sin(k/2)|$$

## What this notebook computes

For each wavenumber $k\in[0,\pi]$ and several initial amplitudes $A$:
1. Simulate the single-cell Bloch oscillator (RK4 + event-based impact)
2. After the startup transient, extract the steady-state frequency $\omega$ via FFT
3. Plot the **amplitude-dependent** nonlinear dispersion $\omega(k, A)$

## Key limitation

The Bloch periodicity assumption requires all unit cells to oscillate
**identically and in phase** — valid for weakly nonlinear regimes but
breaks down for strongly nonlinear or spatially localised dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator

from bloch_dispersion import (
    simulate_bloch_cell,
    extract_frequency,
    bloch_dispersion_scan,
    linear_dispersion,
    plot_bloch_bands,
)

print('bloch_dispersion module loaded.')

## Physical parameters

Matching `PINN_ndofs_nmasses/pinn_ndof_chain_sim.ipynb`.

In [ ]:
mx = 1.0      # primary mass
my = 0.1      # impactor mass
K  = 100.0    # inter-cell spring stiffness
c  = 0.5      # damping coefficient (proportional damping: c_cell = c/K * K_b)
D  = 1.0      # impact gap
r  = 0.7      # coefficient of restitution

omega_lin_max = 2.0 * np.sqrt(K / mx)   # top of acoustic branch
print(f'Linear ω_max = {omega_lin_max:.4f} rad/s')

## Single-cell demo — orbit at $k = \pi/2$

Verify the simulation before the full scan.

In [ ]:
k_demo = np.pi / 2.0
A_demo = 1.5 * D          # amplitude above the gap → impacts occur

t_d, x_d, y_d = simulate_bloch_cell(
    k_demo, mx, my, K, D, r, c=c,
    x0=A_demo, xt0=0.0, y0=0.0, yt0_mag=0.5,
    T_sim=80.0, n_steps=8000,
)

om_demo, amp_demo = extract_frequency(t_d, x_d, skip_transient=0.4)
om_lin_demo       = linear_dispersion(k_demo, K, mx)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t_d, x_d, 'b', lw=0.8, label='Primary x(t)')
axes[0].plot(t_d, y_d, 'r', lw=0.8, label='Impactor y(t)')
axes[0].axhline( D, color='k', lw=0.8, ls='--', label='±D')
axes[0].axhline(-D, color='k', lw=0.8, ls='--')
axes[0].set_ylabel('Displacement')
axes[0].legend(fontsize=9)
axes[0].set_title(f'Bloch cell  k/π={k_demo/np.pi:.2f},  A/D={A_demo/D:.2f}')

rel = x_d - y_d
axes[1].plot(t_d, rel, 'g', lw=0.8)
axes[1].axhline( D, color='k', lw=0.8, ls='--')
axes[1].axhline(-D, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('x − y  (relative)')

plt.tight_layout()
plt.show()

print(f'Steady-state ω = {om_demo:.4f} rad/s')
print(f'Linear      ω = {om_lin_demo:.4f} rad/s')
print(f'Shift         = {100*(om_demo-om_lin_demo)/om_lin_demo:+.2f} %')

## Full Bloch dispersion scan

Scan over $k\in[0,\pi]$ and several amplitudes $A$.  
> **Runtime note:** Each (k, A) pair requires one simulation of length `T_sim`.
> With `N_k=40` and 4 amplitudes this is 160 simulations — typically a few minutes.

In [ ]:
amplitudes = [0.5 * D, 1.0 * D, 1.5 * D, 2.0 * D]   # A/D = 0.5, 1.0, 1.5, 2.0

k_vals, bloch_results = bloch_dispersion_scan(
    mx, my, K, D, r,
    amplitudes=amplitudes,
    c=c,
    N_k=40,
    T_sim=300.0,
    n_steps=30000,
    yt0_mag=0.5,
    skip_transient=0.5,
    verbose=True,
)

print('\nScan complete.')

## Plot: amplitude-dependent dispersion curves

In [ ]:
plot_bloch_bands(
    k_vals, bloch_results,
    K_spring=K, mx=mx,
    omega_max=1.3 * 2.0 * np.sqrt(K / mx),
    save_path='bloch_dispersion_bands.png',
    figsize=(8, 5),
    title=f'Bloch dispersion  —  D={D}, r={r}, my/mx={my/mx}',
)

## Frequency shift vs amplitude

For fixed $k = \pi$ (zone boundary) show how $\omega$ changes with $A/D$.

In [ ]:
A_sweep   = np.linspace(0.1 * D, 3.0 * D, 30)
k_fixed   = np.pi
om_fixed  = np.full(len(A_sweep), np.nan)
om_lin_zb = linear_dispersion(k_fixed, K, mx)

for i, A in enumerate(A_sweep):
    t_s, x_s, _ = simulate_bloch_cell(
        k_fixed, mx, my, K, D, r, c=c,
        x0=A, xt0=0.0, T_sim=300.0, n_steps=30000,
    )
    om_fixed[i], _ = extract_frequency(t_s, x_s, skip_transient=0.5)

mpl.rcParams.update({'font.family': 'Times New Roman', 'pdf.fonttype': 42})
FS = 18
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(A_sweep / D, om_fixed, 'o-', color='C1', lw=2, ms=5,
        label=r'PINN Bloch ($k=\pi$)')
ax.axhline(om_lin_zb, color='k', lw=1.5, ls='--', label='Linear')
ax.axvline(1.0, color='gray', lw=1, ls=':', label='$A/D=1$  (onset of impact)')
ax.set_xlabel(r'Normalised amplitude  $A/D$', fontsize=FS)
ax.set_ylabel(r'$\omega$  (rad/s)', fontsize=FS)
ax.tick_params(labelsize=FS - 3)
ax.legend(fontsize=FS - 5)
ax.set_title(r'Frequency shift at zone boundary $k=\pi$', fontsize=FS - 3)
plt.tight_layout()
plt.savefig('bloch_freq_vs_amplitude.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Linear ω at k=π: {om_lin_zb:.4f} rad/s')
print(f'Nonlinear ω range: {np.nanmin(om_fixed):.4f} – {np.nanmax(om_fixed):.4f} rad/s')

## Tabulate results

In [ ]:
print(f"{'k/π':>6}  {'ω_lin':>9}", end='')
for res in bloch_results:
    print(f"  {'ω ('+res['label']+')':>14}", end='')
print()

for j, k in enumerate(k_vals[1:], 1):     # skip k=0
    ol = linear_dispersion(k, K, mx)
    print(f"{k/np.pi:>6.3f}  {ol:>9.4f}", end='')
    for res in bloch_results:
        ov = res['omega'][j]
        print(f"  {ov:>14.4f}" if not np.isnan(ov) else f"  {'NaN':>14}", end='')
    print()